# AIML Lab Assignment #7 — Overfitting Control (MLP Version)
**Course:** CSET301 | **Dataset:** MNIST | **Semester:** 4th Even 2025

---

### What is this notebook about?
We will train a **MLP (Multi-Layer Perceptron)** — also called a simple neural network — on the MNIST handwritten digit dataset.

We will learn:
- What overfitting is and how to see it
- How to fix overfitting using 4 techniques:
  1. L2 Regularization
  2. Dropout
  3. Early Stopping
  4. Data Augmentation

### What is MLP?
MLP = Multi-Layer Perceptron. It is the simplest type of neural network.
- It takes a flat list of numbers as input
- Passes them through multiple Dense (fully connected) layers
- Each layer learns patterns from the data
- The last layer gives probabilities for each class (digit 0–9)

```
Input (784 pixels) → Dense Layer → Dense Layer → Output (10 classes)
```

---
## Step 0: Import Libraries
First we import all the tools we need. Think of this like gathering all your equipment before starting an experiment.

In [ ]:
# ============================================================
# IMPORT ALL LIBRARIES
# ============================================================

# numpy  → for working with numbers and arrays
import numpy as np

# matplotlib → for drawing graphs and charts
import matplotlib.pyplot as plt

# seaborn → for prettier charts (especially confusion matrix)
import seaborn as sns

# tensorflow & keras → for building and training neural networks
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers         # for L2 regularization
from tensorflow.keras.callbacks import EarlyStopping  # for early stopping

# sklearn → for splitting data and calculating metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# hide unnecessary warning messages
import warnings
warnings.filterwarnings('ignore')

# Check tensorflow version
print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")

---
## Task 1: Data Exploration
### What is MNIST?
MNIST is a famous dataset of **70,000 handwritten digit images**.
- Each image is **28 × 28 pixels** (grayscale)
- Each image belongs to one of **10 classes** (digits 0 to 9)
- It is widely used for learning and testing machine learning models

Let's load it and take a look!

In [ ]:
# ============================================================
# LOAD THE MNIST DATASET
# ============================================================

# keras gives us MNIST directly — no need to download manually
# It returns 2 groups: training data and test data
(X_train_full, y_train_full), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()

# Print basic information about the dataset
print("===== MNIST Dataset Information =====")
print()
print(f"Training images shape : {X_train_full.shape}")
print(f"  → 60,000 images, each 28x28 pixels")
print()
print(f"Training labels shape : {y_train_full.shape}")
print(f"  → 60,000 labels (one number 0-9 per image)")
print()
print(f"Test images shape     : {X_test_raw.shape}")
print(f"  → 10,000 images for final testing")
print()
print(f"Pixel value range     : {X_train_full.min()} to {X_train_full.max()}")
print(f"  → 0 = black, 255 = white")
print()
print(f"Number of classes     : {len(np.unique(y_train_full))}")
print(f"Classes               : {np.unique(y_train_full)}")

In [ ]:
# ============================================================
# VISUALIZE SAMPLE IMAGES
# ============================================================

# Show 2 example images for each digit (0 to 9)
fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle('Sample Images from MNIST Dataset (2 examples per digit)', 
             fontsize=13, fontweight='bold')

for digit in range(10):
    # Find index of first image with this digit label
    first_index  = np.where(y_train_full == digit)[0][0]
    second_index = np.where(y_train_full == digit)[0][1]
    
    # Show first example in top row
    axes[0, digit].imshow(X_train_full[first_index], cmap='gray')
    axes[0, digit].set_title(f'Digit {digit}', fontsize=9)
    axes[0, digit].axis('off')  # hide axis ticks
    
    # Show second example in bottom row
    axes[1, digit].imshow(X_train_full[second_index], cmap='gray')
    axes[1, digit].axis('off')

plt.tight_layout()
plt.show()
print("These are what the handwritten digits look like!")

In [ ]:
# ============================================================
# VISUALIZE CLASS DISTRIBUTION
# ============================================================
# We want to check: are there roughly equal numbers of each digit?
# If one digit has way more images, the model will be biased toward it.

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Count how many images of each digit exist
unique_classes, train_counts = np.unique(y_train_full, return_counts=True)
unique_classes, test_counts  = np.unique(y_test_raw,   return_counts=True)

# Bar chart for training set
axes[0].bar(unique_classes, train_counts, color='steelblue', edgecolor='black', width=0.6)
axes[0].set_title('Training Set — How many images per digit?', fontweight='bold')
axes[0].set_xlabel('Digit (0 to 9)')
axes[0].set_ylabel('Number of Images')
axes[0].set_xticks(unique_classes)
# Write the count on top of each bar
for i, count in zip(unique_classes, train_counts):
    axes[0].text(i, count + 50, str(count), ha='center', fontsize=8)

# Bar chart for test set
axes[1].bar(unique_classes, test_counts, color='coral', edgecolor='black', width=0.6)
axes[1].set_title('Test Set — How many images per digit?', fontweight='bold')
axes[1].set_xlabel('Digit (0 to 9)')
axes[1].set_ylabel('Number of Images')
axes[1].set_xticks(unique_classes)
for i, count in zip(unique_classes, test_counts):
    axes[1].text(i, count + 5, str(count), ha='center', fontsize=8)

plt.tight_layout()
plt.show()
print("Good! The dataset is balanced — roughly equal images per digit.")

---
## Task 2: Data Preparation
Before training, we must:
1. **Normalize** pixel values from 0–255 → 0–1 (makes training faster and stable)
2. **Flatten** images from 28×28 → 784 numbers (MLP needs flat input)
3. **Split** data into Train (60%) / Validation (20%) / Test (20%)

### Why Flatten?
MLP cannot work with 2D images directly. We must convert each 28×28 image into a 1D list of 784 numbers.
```
[28 rows × 28 cols] → [784 numbers in a single row]
```

In [ ]:
# ============================================================
# STEP 1: NORMALIZE pixel values from 0-255 to 0-1
# ============================================================
# Why? Neural networks train much better with small numbers (0 to 1)
# Dividing by 255.0 scales every pixel into this range

X_all_norm  = X_train_full.astype('float32') / 255.0   # training images normalized
X_test_norm = X_test_raw.astype('float32')  / 255.0    # test images normalized

print("Before normalization — pixel range:", X_train_full.min(), "to", X_train_full.max())
print("After  normalization — pixel range:", X_all_norm.min(),  "to", X_all_norm.max())

In [ ]:
# ============================================================
# STEP 2: FLATTEN images from 28x28 to 784 numbers
# ============================================================
# MLP needs a flat 1D array as input, not a 2D image
# reshape(-1, 784) means: keep number of images same (-1), make each image 784 long

X_all_flat  = X_all_norm.reshape(-1, 784)    # shape: (60000, 784)
X_test_flat = X_test_norm.reshape(-1, 784)   # shape: (10000, 784)

print("Image shape before flattening:", X_all_norm.shape)   # (60000, 28, 28)
print("Image shape after  flattening:", X_all_flat.shape)   # (60000, 784)
print()
print("28 x 28 =", 28*28, "← this is why each image becomes 784 numbers")

In [ ]:
# ============================================================
# STEP 3: SPLIT into Train / Validation / Test sets (60-20-20)
# ============================================================
# Why 3 sets?
#   Train      → model learns from this
#   Validation → we check model during training (tune hyperparameters)
#   Test       → final score on completely unseen data

# Combine all data
X_combined = np.concatenate([X_all_flat,   X_test_flat],   axis=0)   # all images
y_combined = np.concatenate([y_train_full, y_test_raw],    axis=0)   # all labels

# First split: 80% for train+val, 20% for test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_combined, y_combined,
    test_size=0.20,        # 20% goes to test
    random_state=42,       # fixed seed → same split every run
    stratify=y_combined    # keep equal class proportions in each split
)

# Second split: from 80%, take 25% as validation → gives us 60% train, 20% val
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25,        # 25% of 80% = 20% of total
    random_state=42,
    stratify=y_trainval
)

total = len(X_combined)
print("===== Data Split Summary =====")
print(f"Total samples     : {total}")
print(f"Training set      : {len(X_train):>6} images  ({len(X_train)/total*100:.1f}%)")
print(f"Validation set    : {len(X_val):>6} images  ({len(X_val)/total*100:.1f}%)")
print(f"Test set          : {len(X_test):>6} images  ({len(X_test)/total*100:.1f}%)")
print()
print(f"Input shape per image : {X_train.shape[1]} numbers (flattened 28x28)")

In [ ]:
# ============================================================
# (OPTIONAL) DATA AUGMENTATION SETUP — for MLP
# ============================================================
# Note: Traditional ImageDataGenerator works with 2D images (CNN)
# For MLP, we do augmentation manually — add small random noise to images
# This makes the training set more diverse

def augment_mlp_data(X, y, num_augmented=10000):
    """
    Create augmented copies by adding small random noise.
    This simulates slight variations in handwriting.
    """
    # Pick random images to augment
    indices = np.random.choice(len(X), size=num_augmented, replace=True)
    X_sample = X[indices]
    y_sample = y[indices]
    
    # Add very small random noise (values between -0.05 and +0.05)
    noise = np.random.uniform(-0.05, 0.05, X_sample.shape)
    X_augmented = np.clip(X_sample + noise, 0.0, 1.0)  # keep values in [0,1]
    
    return X_augmented, y_sample

# Create augmented data
X_aug, y_aug = augment_mlp_data(X_train, y_train, num_augmented=10000)

# Combine original training + augmented data
X_train_aug = np.concatenate([X_train, X_aug], axis=0)
y_train_aug = np.concatenate([y_train, y_aug], axis=0)

print("Original training size :", len(X_train))
print("Augmented samples added:", len(X_aug))
print("Total training size    :", len(X_train_aug))
print()
print("Augmentation adds noise to simulate more varied handwriting!")

---
## Task 3: Baseline MLP Model (Without Any Overfitting Control)
### What is Overfitting?
Overfitting happens when the model **memorizes** training data instead of **learning** patterns.
- Training accuracy keeps going UP
- Validation accuracy STOPS improving or goes DOWN

We build a model WITHOUT any safeguards so we can clearly SEE overfitting.

In [ ]:
# ============================================================
# BUILD THE BASELINE MLP MODEL
# ============================================================

def build_baseline_mlp():
    """
    Simple MLP — no regularization, no dropout.
    Intentionally large so it can overfit.
    
    Architecture:
    Input (784) → Dense(512) → Dense(256) → Dense(128) → Output(10)
    """
    model = keras.Sequential([
        
        # Input layer: 784 numbers (flattened 28x28 image)
        layers.Input(shape=(784,)),
        
        # Hidden Layer 1: 512 neurons, ReLU activation
        # ReLU = max(0, x) → makes output 0 if negative, keeps positive values
        layers.Dense(512, activation='relu'),
        
        # Hidden Layer 2: 256 neurons
        layers.Dense(256, activation='relu'),
        
        # Hidden Layer 3: 128 neurons
        layers.Dense(128, activation='relu'),
        
        # Output Layer: 10 neurons (one per digit class)
        # Softmax converts raw scores into probabilities that add up to 1
        layers.Dense(10, activation='softmax')
        
    ], name='Baseline_MLP')
    
    return model

# Create the model
baseline_model = build_baseline_mlp()

# Show model structure
baseline_model.summary()
print()
print("Params = weights the model learns. More params → more powerful but more overfitting risk.")

In [ ]:
# ============================================================
# COMPILE THE MODEL — set optimizer, loss, and metric
# ============================================================

baseline_model.compile(
    
    # Optimizer: Adam adjusts learning speed automatically
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    
    # Loss function: measures how wrong the model is
    # sparse_categorical_crossentropy is used when labels are integers (0,1,2...)
    loss='sparse_categorical_crossentropy',
    
    # Metric: we want to track accuracy
    metrics=['accuracy']
)

print("Model compiled!")
print("Optimizer : Adam (learning_rate=0.001)")
print("Loss      : sparse_categorical_crossentropy")
print("Metric    : accuracy")

In [ ]:
# ============================================================
# TRAIN THE BASELINE MODEL — 30 epochs to see overfitting
# ============================================================

# epoch = one full pass through ALL training images
# batch_size = how many images processed at once before updating weights

print("Training Baseline MLP... (no overfitting control)")
print("Watch: training accuracy goes up, but validation accuracy may diverge!")
print()

history_baseline = baseline_model.fit(
    X_train, y_train,           # training data
    epochs=30,                  # train for 30 rounds
    batch_size=64,              # process 64 images at a time
    validation_data=(X_val, y_val),   # check on val set after each epoch
    verbose=1                   # show progress
)

# Evaluate on test set
baseline_test_loss, baseline_test_acc = baseline_model.evaluate(X_test, y_test, verbose=0)
print()
print(f"Baseline Test Accuracy : {baseline_test_acc*100:.2f}%")
print(f"Baseline Test Loss     : {baseline_test_loss:.4f}")

In [ ]:
# ============================================================
# HELPER FUNCTION: Plot Training vs Validation curves
# ============================================================
# We use this function for EVERY model to see overfitting visually

def plot_training_curves(history, title='Training History'):
    """
    Plots accuracy and loss for training and validation.
    If train and val curves diverge → overfitting!
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    epochs = range(1, len(history.history['accuracy']) + 1)
    
    # --- Left chart: Accuracy ---
    axes[0].plot(epochs, history.history['accuracy'],     
                 'b-o', markersize=3, label='Training Accuracy')
    axes[0].plot(epochs, history.history['val_accuracy'], 
                 'r-o', markersize=3, label='Validation Accuracy')
    axes[0].set_title(f'{title}\nAccuracy', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # --- Right chart: Loss ---
    axes[1].plot(epochs, history.history['loss'],     
                 'b-o', markersize=3, label='Training Loss')
    axes[1].plot(epochs, history.history['val_loss'], 
                 'r-o', markersize=3, label='Validation Loss')
    axes[1].set_title(f'{title}\nLoss', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot the baseline model curves
plot_training_curves(history_baseline, title='Baseline MLP (Overfitting Expected)')

# Calculate and show the overfitting gap
train_acc_last = history_baseline.history['accuracy'][-1]
val_acc_last   = history_baseline.history['val_accuracy'][-1]
gap = (train_acc_last - val_acc_last) * 100

print(f"Final Training Accuracy   : {train_acc_last*100:.2f}%")
print(f"Final Validation Accuracy : {val_acc_last*100:.2f}%")
print(f"Overfitting Gap           : {gap:.2f}%")
if gap > 3:
    print("→ OVERFITTING DETECTED! Gap is large.")
else:
    print("→ Gap is small. Minimal overfitting.")

---
## Task 4: Overfitting Control Techniques

Now we apply 4 techniques one by one to reduce overfitting.

---
### Technique 4.1 — L2 Regularization
**What it does:** Adds a penalty to the loss for large weights. Forces the model to use small weights → simpler model → less overfitting.

**Think of it like:** A teacher penalizing a student for overly complicated answers. Simple answers are rewarded.

In [ ]:
# ============================================================
# TECHNIQUE 1: L2 REGULARIZATION
# ============================================================

def build_mlp_with_l2(l2_lambda=0.001):
    """
    MLP with L2 regularization on every Dense layer.
    l2_lambda = how strong the penalty is (0.001 is a good start)
    """
    # Create the regularizer object with our chosen strength
    l2_reg = regularizers.l2(l2_lambda)
    
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        
        # kernel_regularizer applies L2 penalty to this layer's weights
        layers.Dense(512, activation='relu', kernel_regularizer=l2_reg),
        layers.Dense(256, activation='relu', kernel_regularizer=l2_reg),
        layers.Dense(128, activation='relu', kernel_regularizer=l2_reg),
        
        # Output layer — no regularization needed here
        layers.Dense(10, activation='softmax')
        
    ], name='L2_Regularization_MLP')
    
    return model

# Build model with L2 lambda = 0.001
l2_model = build_mlp_with_l2(l2_lambda=0.001)

l2_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Training L2 Regularization MLP...")
print()

history_l2 = l2_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    verbose=1
)

l2_test_loss, l2_test_acc = l2_model.evaluate(X_test, y_test, verbose=0)
print()
print(f"L2 Model Test Accuracy : {l2_test_acc*100:.2f}%")

In [ ]:
# Plot training curves for L2 model
plot_training_curves(history_l2, title='L2 Regularization MLP (lambda=0.001)')

# Show gap
train_acc_l2 = history_l2.history['accuracy'][-1]
val_acc_l2   = history_l2.history['val_accuracy'][-1]
print(f"Training Accuracy   : {train_acc_l2*100:.2f}%")
print(f"Validation Accuracy : {val_acc_l2*100:.2f}%")
print(f"Overfitting Gap     : {(train_acc_l2 - val_acc_l2)*100:.2f}%")

---
### Technique 4.2 — Dropout
**What it does:** During training, randomly turns off (sets to zero) some neurons in each step. The model cannot rely on specific neurons, so it learns more robust patterns.

**Think of it like:** A sports team practicing with some players randomly sitting out. The remaining players are forced to be more versatile.

In [ ]:
# ============================================================
# TECHNIQUE 2: DROPOUT
# ============================================================

def build_mlp_with_dropout(dropout_rate=0.3):
    """
    MLP with Dropout layers.
    dropout_rate = fraction of neurons to randomly turn off
    0.3 means 30% of neurons are turned off in each training step
    """
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        
        layers.Dense(512, activation='relu'),
        # Dropout AFTER the Dense layer
        # 30% of the 512 neurons will be randomly turned off
        layers.Dropout(dropout_rate),
        
        layers.Dense(256, activation='relu'),
        layers.Dropout(dropout_rate),
        
        layers.Dense(128, activation='relu'),
        layers.Dropout(dropout_rate),
        
        # No dropout on output layer
        layers.Dense(10, activation='softmax')
        
    ], name=f'Dropout_{dropout_rate}_MLP')
    
    return model

# Build with dropout rate = 0.3 (30%)
dropout_model = build_mlp_with_dropout(dropout_rate=0.3)

dropout_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Training Dropout MLP (rate = 0.3)...")
print()

history_dropout = dropout_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    verbose=1
)

dropout_test_loss, dropout_test_acc = dropout_model.evaluate(X_test, y_test, verbose=0)
print()
print(f"Dropout Model Test Accuracy : {dropout_test_acc*100:.2f}%")

In [ ]:
# Plot training curves for Dropout model
plot_training_curves(history_dropout, title='Dropout MLP (rate=0.3)')

# Show gap
train_acc_do = history_dropout.history['accuracy'][-1]
val_acc_do   = history_dropout.history['val_accuracy'][-1]
print(f"Training Accuracy   : {train_acc_do*100:.2f}%")
print(f"Validation Accuracy : {val_acc_do*100:.2f}%")
print(f"Overfitting Gap     : {(train_acc_do - val_acc_do)*100:.2f}%")

---
### Technique 4.3 — Early Stopping
**What it does:** Monitors validation loss after every epoch. If it stops improving for several epochs, training is automatically stopped and the best model is restored.

**Think of it like:** Stopping a student from studying after they've already mastered the topic — more studying would just cause over-memorization.

In [ ]:
# ============================================================
# TECHNIQUE 3: EARLY STOPPING
# ============================================================

# Build same baseline model — no regularization
# Early stopping alone will control overfitting
early_stop_model = build_baseline_mlp()
early_stop_model._name = 'EarlyStopping_MLP'

early_stop_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Create Early Stopping callback
early_stop_callback = EarlyStopping(
    monitor='val_loss',       # watch validation loss
    patience=5,               # stop after 5 epochs of no improvement
    restore_best_weights=True,# go back to the best epoch's weights
    verbose=1                 # print message when stopping
)

print("Training with Early Stopping...")
print("Max epochs allowed = 50, but training will stop early if val_loss stops improving!")
print()

history_es = early_stop_model.fit(
    X_train, y_train,
    epochs=50,                          # allow up to 50 epochs
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop_callback],    # pass the callback here
    verbose=1
)

es_test_loss, es_test_acc = early_stop_model.evaluate(X_test, y_test, verbose=0)
print()
print(f"Early Stopping Test Accuracy : {es_test_acc*100:.2f}%")
print(f"Training stopped at epoch    : {len(history_es.history['loss'])} (out of max 50)")

In [ ]:
# Plot training curves for Early Stopping model
plot_training_curves(history_es, title='Early Stopping MLP (patience=5)')

# Show gap
train_acc_es = history_es.history['accuracy'][-1]
val_acc_es   = history_es.history['val_accuracy'][-1]
print(f"Training Accuracy   : {train_acc_es*100:.2f}%")
print(f"Validation Accuracy : {val_acc_es*100:.2f}%")
print(f"Overfitting Gap     : {(train_acc_es - val_acc_es)*100:.2f}%")
print(f"Total epochs run    : {len(history_es.history['loss'])}")

---
### Technique 4.4 — Data Augmentation
**What it does:** Creates slightly modified versions of training images (with noise). The model sees more variety, so it learns general patterns instead of memorizing.

**Think of it like:** Studying from many different textbooks instead of just one — your understanding becomes more general.

In [ ]:
# ============================================================
# TECHNIQUE 4: DATA AUGMENTATION
# ============================================================

# Build same baseline model
aug_model = build_baseline_mlp()
aug_model._name = 'DataAugmentation_MLP'

aug_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Training with Data Augmentation...")
print(f"Original training size : {len(X_train)}")
print(f"Augmented training size: {len(X_train_aug)}")
print()

# Train on AUGMENTED data (original + noisy copies)
history_aug = aug_model.fit(
    X_train_aug, y_train_aug,   # use augmented training data
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),   # validation is NOT augmented
    verbose=1
)

aug_test_loss, aug_test_acc = aug_model.evaluate(X_test, y_test, verbose=0)
print()
print(f"Data Augmentation Test Accuracy : {aug_test_acc*100:.2f}%")

In [ ]:
# Plot training curves for Augmentation model
plot_training_curves(history_aug, title='Data Augmentation MLP')

# Show gap
train_acc_aug = history_aug.history['accuracy'][-1]
val_acc_aug   = history_aug.history['val_accuracy'][-1]
print(f"Training Accuracy   : {train_acc_aug*100:.2f}%")
print(f"Validation Accuracy : {val_acc_aug*100:.2f}%")
print(f"Overfitting Gap     : {(train_acc_aug - val_acc_aug)*100:.2f}%")

---
## Task 5: Hyperparameter Tuning
We test different values of:
- Dropout rates: 0.2, 0.3, 0.5
- L2 lambda values: 0.0001, 0.001, 0.01
- Batch sizes and learning rates

**Goal:** Find the combination that gives highest validation accuracy with smallest overfitting gap.

In [ ]:
# ============================================================
# EXPERIMENT A: Different Dropout Rates
# ============================================================

dropout_rates_to_test = [0.2, 0.3, 0.5]
dropout_experiment_results = {}

for rate in dropout_rates_to_test:
    print(f"\nTraining with Dropout rate = {rate} ...")
    
    # Build and compile model
    m = build_mlp_with_dropout(dropout_rate=rate)
    m.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    
    # Train for 20 epochs (faster experiment)
    h = m.fit(X_train, y_train,
              epochs=20,
              batch_size=64,
              validation_data=(X_val, y_val),
              verbose=0)    # verbose=0 → don't print each epoch
    
    # Get test accuracy and overfitting gap
    _, test_acc = m.evaluate(X_test, y_test, verbose=0)
    gap = h.history['accuracy'][-1] - h.history['val_accuracy'][-1]
    
    dropout_experiment_results[rate] = {
        'history'   : h,
        'test_acc'  : test_acc,
        'overfit_gap': gap
    }
    
    print(f"  Dropout={rate} | Test Acc={test_acc*100:.2f}% | Overfit Gap={gap*100:.2f}%")

# Compare on a chart
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Dropout Rate Comparison', fontweight='bold')
colors = ['green', 'blue', 'red']

for rate, color in zip(dropout_rates_to_test, colors):
    h = dropout_experiment_results[rate]['history']
    epochs = range(1, len(h.history['val_accuracy']) + 1)
    axes[0].plot(epochs, h.history['val_accuracy'], color=color, label=f'Dropout={rate}')
    axes[1].plot(epochs, h.history['val_loss'],     color=color, label=f'Dropout={rate}')

axes[0].set_title('Validation Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Validation Loss');     axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EXPERIMENT B: Different L2 Lambda Values
# ============================================================

l2_lambdas_to_test = [0.0001, 0.001, 0.01]
l2_experiment_results = {}

for lam in l2_lambdas_to_test:
    print(f"\nTraining with L2 lambda = {lam} ...")
    
    m = build_mlp_with_l2(l2_lambda=lam)
    m.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    
    h = m.fit(X_train, y_train,
              epochs=20, batch_size=64,
              validation_data=(X_val, y_val),
              verbose=0)
    
    _, test_acc = m.evaluate(X_test, y_test, verbose=0)
    gap = h.history['accuracy'][-1] - h.history['val_accuracy'][-1]
    l2_experiment_results[lam] = {'history': h, 'test_acc': test_acc, 'overfit_gap': gap}
    print(f"  L2={lam} | Test Acc={test_acc*100:.2f}% | Overfit Gap={gap*100:.2f}%")

# Compare on a chart
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('L2 Lambda Comparison', fontweight='bold')
colors = ['green', 'blue', 'red']

for lam, color in zip(l2_lambdas_to_test, colors):
    h = l2_experiment_results[lam]['history']
    epochs = range(1, len(h.history['val_accuracy']) + 1)
    axes[0].plot(epochs, h.history['val_accuracy'], color=color, label=f'λ={lam}')
    axes[1].plot(epochs, h.history['val_loss'],     color=color, label=f'λ={lam}')

axes[0].set_title('Validation Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Validation Loss');     axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EXPERIMENT C: Different Batch Sizes and Learning Rates
# ============================================================

# Different configurations to test
configs = [
    {'batch_size': 32,  'lr': 0.001,  'label': 'batch=32,  lr=0.001'},
    {'batch_size': 64,  'lr': 0.001,  'label': 'batch=64,  lr=0.001'},
    {'batch_size': 128, 'lr': 0.001,  'label': 'batch=128, lr=0.001'},
    {'batch_size': 64,  'lr': 0.0001, 'label': 'batch=64,  lr=0.0001'},
]

config_results = []

for cfg in configs:
    print(f"\nTesting: {cfg['label']}")
    
    m = build_mlp_with_dropout(dropout_rate=0.3)
    m.compile(optimizer=keras.optimizers.Adam(cfg['lr']),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    
    h = m.fit(X_train, y_train,
              epochs=15, batch_size=cfg['batch_size'],
              validation_data=(X_val, y_val),
              verbose=0)
    
    _, test_acc = m.evaluate(X_test, y_test, verbose=0)
    best_val    = max(h.history['val_accuracy'])
    config_results.append({'label': cfg['label'], 'test_acc': test_acc, 'best_val': best_val})
    print(f"  Test Acc={test_acc*100:.2f}% | Best Val Acc={best_val*100:.2f}%")

# Summary table
print()
print("===== Hyperparameter Tuning Summary =====")
print(f"{'Configuration':<30} {'Test Acc':>10} {'Best Val Acc':>13}")
print("-" * 55)
for r in config_results:
    print(f"{r['label']:<30} {r['test_acc']*100:>9.2f}% {r['best_val']*100:>12.2f}%")

---
## Additional Task: Combined Techniques (Best Model)
Now we combine **ALL techniques together**:
- L2 Regularization + Dropout + Early Stopping + Data Augmentation

This should give us the lowest overfitting and the best accuracy!

In [ ]:
# ============================================================
# COMBINED BEST MODEL: L2 + Dropout + Early Stopping + Augmentation
# ============================================================

def build_best_mlp():
    """
    Best MLP combining L2 regularization AND Dropout together.
    """
    l2_reg = regularizers.l2(0.0005)   # mild L2 regularization
    
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        
        layers.Dense(512, activation='relu', kernel_regularizer=l2_reg),
        layers.Dropout(0.3),    # 30% dropout
        
        layers.Dense(256, activation='relu', kernel_regularizer=l2_reg),
        layers.Dropout(0.3),
        
        layers.Dense(128, activation='relu', kernel_regularizer=l2_reg),
        layers.Dropout(0.2),    # slightly lower dropout in last hidden layer
        
        layers.Dense(10, activation='softmax')
        
    ], name='Combined_Best_MLP')
    
    return model

# Build the best model
best_model = build_best_mlp()

best_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early Stopping callback
best_es_callback = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

print("Training COMBINED BEST MODEL...")
print("Techniques: L2 Regularization + Dropout + Early Stopping + Data Augmentation")
print()

# Train on augmented data with early stopping
history_combined = best_model.fit(
    X_train_aug, y_train_aug,    # augmented training data
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[best_es_callback],
    verbose=1
)

combined_test_loss, combined_test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print()
print(f"BEST MODEL Test Accuracy : {combined_test_acc*100:.2f}%")
print(f"BEST MODEL Test Loss     : {combined_test_loss:.4f}")
print(f"Total epochs trained     : {len(history_combined.history['loss'])}")

In [ ]:
# Plot training curves for Combined model
plot_training_curves(history_combined, 
                     title='Combined Best MLP (L2 + Dropout + EarlyStopping + Augmentation)')

# Show gap
train_acc_comb = history_combined.history['accuracy'][-1]
val_acc_comb   = history_combined.history['val_accuracy'][-1]
print(f"Training Accuracy   : {train_acc_comb*100:.2f}%")
print(f"Validation Accuracy : {val_acc_comb*100:.2f}%")
print(f"Overfitting Gap     : {(train_acc_comb - val_acc_comb)*100:.2f}%")

---
## Task 6: Performance Evaluation
### 6.1 — Compare All Models Side by Side

In [ ]:
# ============================================================
# COMPARE ALL 6 MODELS
# ============================================================

# List of all models and their training histories
all_models = [
    ('Baseline (No Control)',     baseline_model,    history_baseline),
    ('L2 Regularization',         l2_model,          history_l2),
    ('Dropout (rate=0.3)',        dropout_model,     history_dropout),
    ('Early Stopping',            early_stop_model,  history_es),
    ('Data Augmentation',         aug_model,         history_aug),
    ('Combined Best Model',       best_model,        history_combined),
]

comparison_results = []

for model_name, model, history in all_models:
    # Get test accuracy
    _, test_acc = model.evaluate(X_test, y_test, verbose=0)
    
    # Get final train and val accuracy
    train_acc_final = history.history['accuracy'][-1]
    val_acc_final   = history.history['val_accuracy'][-1]
    gap = train_acc_final - val_acc_final
    
    comparison_results.append({
        'Model'      : model_name,
        'Train Acc'  : train_acc_final,
        'Val Acc'    : val_acc_final,
        'Test Acc'   : test_acc,
        'Overfit Gap': gap
    })

# Print comparison table
print("=" * 75)
print(f"{'Model':<28} {'Train Acc':>10} {'Val Acc':>9} {'Test Acc':>9} {'Gap':>8}")
print("-" * 75)
for r in comparison_results:
    marker = " ★" if r['Model'] == 'Combined Best Model' else ""
    print(f"{r['Model']:<28} {r['Train Acc']*100:>9.2f}% "
          f"{r['Val Acc']*100:>8.2f}% "
          f"{r['Test Acc']*100:>8.2f}% "
          f"{r['Overfit Gap']*100:>7.2f}%{marker}")
print("=" * 75)
print("★ = Best model | Gap = Train Acc - Val Acc (lower gap = less overfitting)")

In [ ]:
# ============================================================
# VISUAL COMPARISON BAR CHART
# ============================================================

model_names = [r['Model'] for r in comparison_results]
test_accs   = [r['Test Acc']    * 100 for r in comparison_results]
gaps        = [r['Overfit Gap'] * 100 for r in comparison_results]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(len(model_names))

# Chart 1: Test Accuracy
bars = axes[0].bar(x, test_accs, color='steelblue', edgecolor='black', width=0.5)
axes[0].set_title('Test Accuracy per Model\n(Higher is Better)', fontweight='bold')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
axes[0].set_ylim([min(test_accs) - 3, 100])
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}%', ha='center', fontsize=8)

# Chart 2: Overfitting Gap (lower = better)
gap_colors = ['red' if g > 2 else 'green' for g in gaps]
bars2 = axes[1].bar(x, gaps, color=gap_colors, edgecolor='black', width=0.5)
axes[1].set_title('Overfitting Gap (Train - Val Accuracy)\n(Lower is Better)', fontweight='bold')
axes[1].set_ylabel('Gap (%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=30, ha='right', fontsize=8)
axes[1].axhline(y=0, color='black', linewidth=0.8, linestyle='--')
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, gaps):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()
print("Green bars = low overfitting | Red bars = high overfitting")

### 6.2 — Classification Report (Best Model)
This shows Precision, Recall, and F1-score for each digit.

In [ ]:
# ============================================================
# CLASSIFICATION REPORT — Best Model
# ============================================================

# Get predictions from the best model
y_pred_probs = best_model.predict(X_test, verbose=0)   # probabilities for each class
y_pred       = np.argmax(y_pred_probs, axis=1)         # take class with highest probability

print("=" * 60)
print("  Classification Report — Combined Best MLP Model")
print("=" * 60)
print()
print(classification_report(
    y_test, y_pred,
    target_names=[f'Digit {i}' for i in range(10)]
))
print()
print("Precision = Out of all images PREDICTED as digit X, how many were actually X?")
print("Recall    = Out of all images THAT ARE digit X, how many did we correctly find?")
print("F1-score  = Balance between Precision and Recall (higher = better)")

### 6.3 — Confusion Matrix (Best Model)
Shows which digits the model confuses most.

In [ ]:
# ============================================================
# CONFUSION MATRIX — Best Model
# ============================================================

# Build the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Draw it as a heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,          # show numbers in each cell
    fmt='d',             # format as integers
    cmap='Blues',        # color scheme
    xticklabels=range(10),
    yticklabels=range(10),
    linewidths=0.5,
    ax=ax
)

ax.set_title('Confusion Matrix — Combined Best MLP Model\n'
             'Diagonal = correct predictions | Off-diagonal = mistakes',
             fontsize=12, fontweight='bold')
ax.set_xlabel('What the model PREDICTED', fontsize=11)
ax.set_ylabel('What the image ACTUALLY is', fontsize=11)

plt.tight_layout()
plt.show()

# Show per-class accuracy
print("\nPer-digit accuracy:")
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100
for digit, acc in enumerate(per_class_acc):
    print(f"  Digit {digit}: {acc:.2f}%")

### 6.4 — All Learning Curves Side by Side

In [ ]:
# ============================================================
# ALL LEARNING CURVES — Side by Side
# ============================================================

histories_to_compare = [
    (history_baseline, 'Baseline (No Control)'),
    (history_l2,       'L2 Regularization'),
    (history_dropout,  'Dropout (0.3)'),
    (history_es,       'Early Stopping'),
    (history_aug,      'Data Augmentation'),
    (history_combined, 'Combined Best Model'),
]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle('Accuracy Learning Curves — All Models\n'
             '(Blue = Training, Red = Validation)', 
             fontsize=13, fontweight='bold')

for idx, (hist, title) in enumerate(histories_to_compare):
    row = idx // 2
    col = idx  % 2
    ax  = axes[row][col]
    
    epochs = range(1, len(hist.history['accuracy']) + 1)
    ax.plot(epochs, hist.history['accuracy'],     'b-', linewidth=2, label='Train')
    ax.plot(epochs, hist.history['val_accuracy'], 'r--', linewidth=2, label='Validation')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim([0.85, 1.01])

plt.tight_layout()
plt.show()
print("If the red line (validation) drops below or diverges from blue (train) → overfitting!")

---
## Discussion: Bias-Variance Trade-off

| Technique | Effect on Bias | Effect on Variance | Result |
|---|---|---|---|
| No Regularization | Low | High | Memorizes data → overfitting |
| L2 Regularization | Slightly Higher | Lower | Simpler model, better generalization |
| Dropout | Slightly Higher | Lower | Forces robust learning |
| Early Stopping | Slightly Higher | Lower | Stops before memorization |
| Data Augmentation | Same | Lower | More variety → better generalization |
| **Combined** | **Balanced** | **Minimized** | **Best overall performance** |

### Key Insight
- **Too little regularization** → model memorizes training data (overfitting = high variance)
- **Too much regularization** → model is too simple to learn anything (underfitting = high bias)
- **Sweet spot** → low bias AND low variance = combining mild versions of multiple techniques

---
## Final Summary

In [ ]:
# ============================================================
# FINAL SUMMARY — Print all results
# ============================================================

print("=" * 70)
print("    AIML Lab Week 7 — MLP Overfitting Control — Final Results")
print("=" * 70)
print()
print(f"{'Model':<28} {'Train':>7} {'Val':>7} {'Test':>7} {'Gap':>7}")
print("-" * 60)
for r in comparison_results:
    star = " ★" if r['Model'] == 'Combined Best Model' else ""
    print(f"{r['Model']:<28} "
          f"{r['Train Acc']*100:>6.1f}% "
          f"{r['Val Acc']*100:>6.1f}% "
          f"{r['Test Acc']*100:>6.1f}% "
          f"{r['Overfit Gap']*100:>6.1f}%{star}")
print("-" * 60)
print()
print("★ Best Model = L2 + Dropout + Data Augmentation + Early Stopping")
print()
print("KEY CONCLUSIONS:")
print("  1. Baseline MLP overfits — large gap between train and val accuracy")
print("  2. Each technique alone reduces the overfitting gap")
print("  3. Combining all techniques gives the BEST result")
print("  4. Early Stopping is the cheapest technique — no extra computation")
print("  5. Data Augmentation is most helpful when labeled data is limited")
print("  6. MLP is simpler than CNN but still achieves good accuracy on MNIST")